# Activité Pratique N°3 — RAG Multimodal Agentique
**Module :** IA Générative Multi Modale — Master 5 IIIA, HESTIM
**Encadrant :** Pr. Mohamed YOUSSFI
**Réalisé par :** Nada Mahrouz

**Objectif :** concevoir une application RAG (Retrieval-Augmented Generation) multimodale et agentique capable
d'extraire, d'indexer et d'interroger de façon autonome le texte et les images de l'article
*Attention Is All You Need* (Vaswani et al., 2017).

**Pile technique :** PyMuPDF (extraction) · CLIP ViT-B-32 / open_clip (embeddings communs texte-image) ·
ChromaDB (base vectorielle) · LangChain tools + LangGraph (agent ReAct) · Ollama (LLM local).

| Étape | Contenu |
|---|---|
| 0 | Installation & configuration |
| 1 | Extraction du contenu multimodal (texte & images) |
| 2 | Vectorisation dans un espace latent commun (CLIP) |
| 3 | Indexation dans ChromaDB (SHA-256, métadonnées) |
| 4 | Recherche multimodale par similarité cosinus |
| 5 | Transformation du retriever en outil agentique |
| 6 | Agent ReAct avec LangGraph (MIND + BODY) |
| 7 | Tests et validation |

## Étape 0 — Installation et configuration

In [ ]:
!pip install -q open_clip_torch chromadb pymupdf langgraph langchain-core langchain-ollama

In [ ]:
# Installation et démarrage d'Ollama (LLM local, sans clé API)
!apt-get -qq install -y zstd > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
import subprocess, time, torch
ollama_proc = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)
LLM_MODEL = "qwen2.5:7b" if torch.cuda.is_available() else "qwen2.5:3b"  # modèles supportant le tool calling
print("GPU :", torch.cuda.is_available(), "| LLM :", LLM_MODEL)
!ollama pull {LLM_MODEL} 2>&1 | tail -n 1

In [ ]:
import os, io, hashlib, json, re
import numpy as np
import torch, open_clip
import pymupdf as fitz  # PyMuPDF
import chromadb
from PIL import Image
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
PDF_PATH = "attention_is_all_you_need.pdf"
IMG_DIR = "extracted_images"
os.makedirs(IMG_DIR, exist_ok=True)

if not os.path.exists(PDF_PATH):
    !wget -q -O {PDF_PATH} https://arxiv.org/pdf/1706.03762v7
print("PDF :", PDF_PATH, os.path.getsize(PDF_PATH), "octets | device :", DEVICE)

## Étape 1 — Extraction du contenu multimodal (Textes & Images)
- **Texte :** extraction page par page avec PyMuPDF, puis découpage en *chunks* courts avec chevauchement.
  CLIP a une fenêtre de contexte textuelle de **77 tokens** : les fragments sont donc limités à ~45 mots
  (chevauchement de 10 mots) pour ne pas être tronqués.
- **Images :** extraction des images *raster* intégrées (`page.get_images`) et sauvegarde en PNG.
  Les figures vectorielles (dessinées en PDF, non rasterisées) sont récupérées en recadrant le rendu de la page
  sur la zone des dessins (`page.get_drawings`).

In [ ]:
doc = fitz.open(PDF_PATH)
print("Nombre de pages :", len(doc))

# --- 1.a Texte par page
pages_text = []
for i, page in enumerate(doc):
    txt = re.sub(r"\s+", " ", page.get_text("text")).strip()
    pages_text.append({"page": i + 1, "text": txt})
print("Total caractères :", sum(len(p["text"]) for p in pages_text))
print("\nAperçu page 1 :\n", pages_text[0]["text"][:400], "...")

In [ ]:
# --- 1.b Découpage en chunks (fenêtre glissante de mots)
def chunk_text(text, size=45, overlap=10):
    words = text.split()
    step = size - overlap
    return [" ".join(words[i:i + size]) for i in range(0, max(len(words) - overlap, 1), step)]

text_chunks = []
for p in pages_text:
    for j, ch in enumerate(chunk_text(p["text"])):
        text_chunks.append({"page": p["page"], "chunk_index": j, "text": ch})

print("Nombre de chunks :", len(text_chunks))
for c in text_chunks[20:23]:
    print(f"[p.{c['page']} #{c['chunk_index']}] {c['text']}\n")

In [ ]:
# --- 1.c Extraction des images (raster) + figures vectorielles
image_records = []
seen = set()
for pno, page in enumerate(doc, start=1):
    # images raster intégrées
    for k, img in enumerate(page.get_images(full=True)):
        xref = img[0]
        base = doc.extract_image(xref)
        h = hashlib.md5(base["image"]).hexdigest()
        if h in seen or base["width"] < 50 or base["height"] < 50:
            continue
        seen.add(h)
        pil = Image.open(io.BytesIO(base["image"])).convert("RGB")
        path = f"{IMG_DIR}/page{pno}_img{k}.png"
        pil.save(path)
        image_records.append({"page": pno, "path": path, "source": "raster", "size": pil.size})
    # figures vectorielles : on recadre la zone englobant les dessins
    drawings = [d["rect"] for d in page.get_drawings() if d["rect"].width > 5 and d["rect"].height > 5]
    if len(drawings) > 30:  # beaucoup de tracés = une figure
        r = drawings[0]
        for d in drawings[1:]:
            r |= d
        if r.width > 100 and r.height > 100:
            pix = page.get_pixmap(clip=r, dpi=150)
            path = f"{IMG_DIR}/page{pno}_vector.png"
            pix.save(path)
            image_records.append({"page": pno, "path": path, "source": "vector", "size": (pix.width, pix.height)})

# Légende associée à chaque figure : phrase "Figure N: ..." présente sur la même page
for r in image_records:
    txt = pages_text[r["page"] - 1]["text"]
    m = re.search(r"Figure \d+:.{0,220}", txt)
    cap = m.group(0) if m else ""
    r["caption"] = cap[:cap.rfind(".") + 1] if "." in cap else cap

print("Images extraites :", len(image_records))
for r in image_records:
    print(r)

In [ ]:
n = len(image_records)
cols = 4
rows = (n + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(16, 4 * rows))
for ax in np.array(axes).ravel():
    ax.axis("off")
for ax, r in zip(np.array(axes).ravel(), image_records):
    ax.imshow(Image.open(r["path"]))
    ax.set_title(f"p.{r['page']} ({r['source']})", fontsize=9)
plt.tight_layout(); plt.show()

## Étape 2 — Vectorisation dans un espace latent commun (CLIP)
CLIP (*Contrastive Language-Image Pre-training*) est entraîné par apprentissage contrastif sur des paires
(image, légende) : un encodeur d'images (ViT-B/32) et un encodeur de texte (Transformer) projettent les deux
modalités dans **le même espace de dimension 512**. Une image et un texte qui la décrit ont alors des vecteurs
proches. On **normalise** chaque vecteur (norme L2 = 1) : le produit scalaire devient égal à la similarité cosinus.

In [ ]:
clip_model, _, clip_preprocess = open_clip.create_model_and_transforms("ViT-B-32", pretrained="laion2b_s34b_b79k")
clip_tokenizer = open_clip.get_tokenizer("ViT-B-32")
clip_model = clip_model.to(DEVICE).eval()

@torch.no_grad()
def embed_texts(texts, batch_size=64):
    out = []
    for i in range(0, len(texts), batch_size):
        tokens = clip_tokenizer(texts[i:i + batch_size]).to(DEVICE)  # tronque à 77 tokens
        feats = clip_model.encode_text(tokens)
        feats = feats / feats.norm(dim=-1, keepdim=True)              # normalisation L2
        out.append(feats.cpu().numpy())
    return np.vstack(out)

@torch.no_grad()
def embed_images(paths):
    imgs = torch.stack([clip_preprocess(Image.open(p).convert("RGB")) for p in paths]).to(DEVICE)
    feats = clip_model.encode_image(imgs)
    feats = feats / feats.norm(dim=-1, keepdim=True)                  # normalisation L2
    return feats.cpu().numpy()

text_embeddings = embed_texts([c["text"] for c in text_chunks])
image_embeddings = embed_images([r["path"] for r in image_records])
print("Embeddings texte :", text_embeddings.shape, "| images :", image_embeddings.shape)
print("Normes (doivent valoir 1) :", np.linalg.norm(text_embeddings[0]).round(4), np.linalg.norm(image_embeddings[0]).round(4))

In [ ]:
# Vérification de l'espace commun : similarité image <-> descriptions textuelles
probes = ["a diagram of the Transformer model architecture with encoder and decoder",
          "scaled dot-product attention and multi-head attention diagram",
          "attention visualization heatmap between words of a sentence",
          "a photo of a cat"]
P = embed_texts(probes)
S = image_embeddings @ P.T
for r, s in zip(image_records, S):
    print(f"{os.path.basename(r['path']):22s}", " | ".join(f"{v:.3f}" for v in s), "-> meilleur :", probes[int(s.argmax())][:45])

## Étape 3 — Indexation et stockage dans ChromaDB
- Base **persistante** ChromaDB, collection configurée en distance **cosinus** (`hnsw:space = cosine`).
- Un **identifiant de document** est calculé par **SHA-256** du contenu binaire du PDF : si le même fichier est
  ré-indexé, on le détecte et on évite les doublons ; s'il a changé, ses anciennes entrées sont supprimées puis remplacées.
- Chaque vecteur est stocké avec ses **métadonnées** : `type` (text / image), `page`, `image_path`, `doc_id`, `source`.

In [ ]:
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(8192), b""):
            h.update(block)
    return h.hexdigest()

DOC_ID = sha256_file(PDF_PATH)
print("doc_id (SHA-256) :", DOC_ID)

chroma_client = chromadb.PersistentClient(path="chroma_db")
try:
    chroma_client.delete_collection("rag_multimodal")   # base neuve pour cette démonstration
except Exception:
    pass
collection = chroma_client.get_or_create_collection(name="rag_multimodal", metadata={"hnsw:space": "cosine"})

def index_document(force=False):
    existing = collection.get(where={"doc_id": DOC_ID}, limit=1)
    if existing["ids"] and not force:
        print("Document déjà indexé (même SHA-256) -> réindexation ignorée.")
        return
    collection.delete(where={"source": os.path.basename(PDF_PATH)})  # supprime une ancienne version éventuelle
    ids, embs, metas, docs = [], [], [], []
    for c, e in zip(text_chunks, text_embeddings):
        ids.append(f"{DOC_ID[:16]}_txt_p{c['page']}_{c['chunk_index']}")
        embs.append(e.tolist()); docs.append(c["text"])
        metas.append({"doc_id": DOC_ID, "source": os.path.basename(PDF_PATH), "type": "text",
                      "page": c["page"], "image_path": ""})
    for k, (r, e) in enumerate(zip(image_records, image_embeddings)):
        ids.append(f"{DOC_ID[:16]}_img_{k}")
        embs.append(e.tolist()); docs.append(f"[IMAGE] page {r['page']} : {r['caption']}")
        metas.append({"doc_id": DOC_ID, "source": os.path.basename(PDF_PATH), "type": "image",
                      "page": r["page"], "image_path": r["path"], "caption": r["caption"]})
    collection.add(ids=ids, embeddings=embs, metadatas=metas, documents=docs)
    print(f"Indexé : {len(text_chunks)} chunks texte + {len(image_records)} images")

index_document()
print("Taille de la collection :", collection.count())
index_document()   # 2e appel : démontre la détection via SHA-256

## Étape 4 — Recherche multimodale (Retrieval)
La fonction `multimodal_search` accepte **soit un texte, soit une image** comme requête : la requête est encodée
par CLIP dans le même espace, puis ChromaDB renvoie les *k* voisins les plus proches au sens du cosinus
(similarité = 1 − distance). On peut filtrer par modalité (`text`, `image` ou les deux).

In [ ]:
def multimodal_search(query=None, image_path=None, k=5, content_type=None):
    if image_path is not None:
        q = embed_images([image_path])[0]
    else:
        q = embed_texts([query])[0]
    where = {"type": content_type} if content_type else None
    res = collection.query(query_embeddings=[q.tolist()], n_results=k, where=where)
    hits = []
    for i in range(len(res["ids"][0])):
        m = res["metadatas"][0][i]
        hits.append({"score": round(1 - res["distances"][0][i], 4), "type": m["type"], "page": m["page"],
                     "image_path": m["image_path"], "caption": m.get("caption", ""),
                     "content": res["documents"][0][i]})
    return hits

def show_results(hits, title=""):
    display(Markdown(f"### {title}"))
    for h in hits:
        if h["type"] == "text":
            print(f"[TEXTE] score={h['score']} | page {h['page']} | {h['content'][:180]}...")
        else:
            print(f"[IMAGE] score={h['score']} | page {h['page']} | {h['image_path']} | {h['caption']}")
            plt.figure(figsize=(4, 3)); plt.imshow(Image.open(h["image_path"])); plt.axis("off"); plt.show()

In [ ]:
show_results(multimodal_search("multi-head attention allows the model to attend to different representation subspaces", k=4, content_type="text"),
             "Requête texte -> textes")

In [ ]:
show_results(multimodal_search("a diagram of the Transformer model architecture with encoder and decoder", k=3, content_type="image"),
             "Requête texte -> images")

In [ ]:
query_img = image_records[0]["path"]
plt.figure(figsize=(3, 3)); plt.imshow(Image.open(query_img)); plt.axis("off"); plt.title("Image requête"); plt.show()
show_results(multimodal_search(image_path=query_img, k=4), "Requête image -> textes + images")

## Étape 5 — Transformation du retriever en outil agentique (Tool)
La recherche vectorielle est encapsulée dans des **outils LangChain** (`@tool`). Le **nom**, la **docstring**
(description) et le **schéma des arguments** sont transmis au LLM : c'est ce qui lui permet de décider *quand*
et *comment* appeler l'outil.
- `retriever_tool` : recherche à partir d'une requête **texte** (textes et/ou images).
- `image_retriever_tool` : recherche à partir d'une **image** (chemin de fichier).

In [ ]:
from langchain_core.tools import tool
from typing import Literal, Optional

def _format_hits(hits):
    lines = []
    for h in hits:
        if h["type"] == "text":
            lines.append(f"- [TEXT | page {h['page']} | score {h['score']}] {h['content']}")
        else:
            lines.append(f"- [IMAGE | page {h['page']} | score {h['score']}] figure file: {h['image_path']} | caption: {h['caption']}")
    return "\n".join(lines) if lines else "No result."

@tool
def retriever_tool(query: str, content_type: Optional[Literal["text", "image", "all"]] = "all", k: int = 5) -> str:
    """Search the paper 'Attention Is All You Need' (Transformer) indexed in a multimodal vector database.
    Use it for ANY question about the Transformer, attention, architecture, training, results or figures.
    Args:
        query: an English search query describing what to look for.
        content_type: 'text' for passages, 'image' for figures/diagrams, 'all' for both.
        k: number of results (default 5).
    Returns passages with their page number and/or figure file paths."""
    if content_type in ("text", "image"):
        return _format_hits(multimodal_search(query, k=k, content_type=content_type))
    # "all" : recherche séparée par modalité puis fusion. Les scores texte->texte de CLIP (~0.7-0.8) sont
    # bien plus élevés que texte->image (~0.25-0.35) (« modality gap ») : une recherche unique ne renverrait
    # jamais d'image. On garde donc les k meilleurs textes + les 2 meilleures images.
    hits = multimodal_search(query, k=k, content_type="text") + multimodal_search(query, k=2, content_type="image")
    return _format_hits(hits)

@tool
def image_retriever_tool(image_path: str, k: int = 5) -> str:
    """Search the paper using an IMAGE as query. ONLY use it when the user explicitly gives the path of an
    existing image file (e.g. 'extracted_images/page4_img0.png'). Never invent a path: to FIND a figure from a
    description, use retriever_tool with content_type='image'. Returns the most similar figures (with their
    caption) and the text passages related to the best matching figure."""
    if not os.path.exists(image_path):
        return (f"Error: image not found '{image_path}'. To find a figure by description, call retriever_tool "
                "with content_type='image'.")
    img_hits = multimodal_search(image_path=image_path, k=max(k, 3), content_type="image")
    best = img_hits[0]
    # chaînage image -> figure la plus proche -> légende -> passages textuels qui l'expliquent
    txt_hits = multimodal_search(best["caption"] or "figure", k=4, content_type="text") if best["caption"] else []
    return "Most similar figures:\n" + _format_hits(img_hits) + "\nRelated passages:\n" + _format_hits(txt_hits)

tools = [retriever_tool, image_retriever_tool]
for t in tools:
    print(t.name, "->", json.dumps(t.args, ensure_ascii=False))
print()
print(retriever_tool.invoke({"query": "positional encoding sine cosine", "content_type": "text", "k": 2}))

## Étape 6 — Construction de l'agent IA (pattern ReAct & LangGraph)
- **MIND (cerveau)** : un LLM local servi par Ollama (Qwen 2.5, qui supporte le *tool calling*), auquel on lie les outils (`bind_tools`).
- **BODY (corps)** : le nœud `tools` (`ToolNode`) qui exécute réellement la recherche multimodale.
- **Cycle ReAct** : *Reasoning* (le LLM réfléchit et décide d'appeler un outil) → *Action* (exécution de l'outil)
  → *Observation* (le résultat revient dans l'état) → … jusqu'à ce que le LLM produise une réponse finale sans appel d'outil.

```
START -> mind --(tool_calls ?)--> tools -> mind -> ... -> END
              \--(pas d'appel)--> END
```

In [ ]:
from typing import Annotated, TypedDict
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage, BaseMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition

SYSTEM_PROMPT = SystemMessage(content=(
    "You are a research assistant specialised in the paper 'Attention Is All You Need'. "
    "Follow the ReAct method: think about what information you need, call the search tools to retrieve it, "
    "read the observations, then answer. ALWAYS use the tools before answering a question about the paper; "
    "never invent facts. Use content_type='image' when the user asks for a figure or diagram. "
    "Use image_retriever_tool ONLY with an image path given by the user; never invent file names. "
    "Break multi-part questions into several searches (one per sub-question). "
    "If a tool returns an error or insufficient information, reformulate and call a tool again. "
    "In the final answer, cite page numbers and give the image file path when a figure is relevant. "
    "Answer in the language of the user."))

class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

llm = ChatOllama(model=LLM_MODEL, temperature=0)
llm_with_tools = llm.bind_tools(tools)                      # MIND

def mind(state: AgentState):
    response = llm_with_tools.invoke([SYSTEM_PROMPT] + state["messages"])
    return {"messages": [response]}

graph = StateGraph(AgentState)
graph.add_node("mind", mind)
graph.add_node("tools", ToolNode(tools))                    # BODY
graph.add_edge(START, "mind")
graph.add_conditional_edges("mind", tools_condition, {"tools": "tools", END: END})
graph.add_edge("tools", "mind")
agent = graph.compile()
from IPython.display import Image as IPImage
try:
    display(IPImage(agent.get_graph().draw_mermaid_png()))
except Exception as e:
    print(agent.get_graph().draw_mermaid())

In [ ]:
def run_agent(question, verbose=True):
    display(Markdown(f"---\n## Question : {question}"))
    final = None
    for step in agent.stream({"messages": [HumanMessage(content=question)]}, stream_mode="updates", config={"recursion_limit": 12}):
        for node, update in step.items():
            for m in update["messages"]:
                if isinstance(m, AIMessage) and m.tool_calls:
                    for tc in m.tool_calls:
                        print(f"[REASONING -> ACTION] {tc['name']}({json.dumps(tc['args'], ensure_ascii=False)})")
                elif isinstance(m, ToolMessage):
                    if verbose:
                        print(f"[OBSERVATION] {m.name} :\n{m.content[:700]}{' ...' if len(m.content) > 700 else ''}\n")
                elif isinstance(m, AIMessage):
                    final = m.content
    display(Markdown(f"**Réponse finale :**\n\n{final}"))
    for p in sorted(set(re.findall(r"extracted_images/[\w\.]+\.png", final or ""))):
        if os.path.exists(p):
            plt.figure(figsize=(4, 3)); plt.imshow(Image.open(p)); plt.axis("off"); plt.title(p); plt.show()
    return final

## Étape 7 — Tests et validation de l'agent RAG
On soumet des requêtes de complexité croissante et on observe le comportement dynamique de l'agent :
décision d'appeler (ou non) un outil, choix de la modalité, récupération des contextes et génération de la réponse.

In [ ]:
# Test 1 : question textuelle factuelle
run_agent("What is scaled dot-product attention and why is it scaled by the square root of d_k?")

In [ ]:
# Test 2 : requête hybride texte + image
run_agent("Montre-moi la figure de l'architecture du Transformer et explique le rôle de l'encodeur et du décodeur.")

In [ ]:
# Test 3 : question nécessitant plusieurs recherches (raisonnement multi-étapes)
run_agent("Compare the BLEU scores of the Transformer on English-German and English-French, and give the training cost of the big model.")

In [ ]:
# Test 4 : requête par image (l'utilisateur fournit une image)
img_q = [r for r in image_records if r["page"] == 4][0]["path"] if any(r["page"] == 4 for r in image_records) else image_records[-1]["path"]
run_agent(f"I have this image: {img_q}. Which part of the paper does it correspond to? Explain the concept it illustrates.")

In [ ]:
# Test 5 : question hors sujet -> l'agent ne devrait pas forcément appeler l'outil
run_agent("Bonjour, qui es-tu et que peux-tu faire ?", verbose=False)

## Analyse et conclusion
- **Extraction :** 15 pages, 39 481 caractères, **178 chunks** ; **6 figures** (3 raster pages 3-4, 3 vectorielles pages 13-15), chacune associée à sa légende « Figure N: … ».
- **Espace commun :** CLIP associe correctement chaque figure à sa description (Figure 1 → « architecture diagram », Figure 2 → « scaled dot-product / multi-head attention », Figures 3-5 → « attention visualization heatmap »), alors que la requête témoin « a photo of a cat » obtient toujours le score le plus faible.
- **Indexation :** 184 vecteurs dans ChromaDB ; le 2e appel à `index_document()` est ignoré grâce à l'empreinte SHA-256.
- **Écart de modalité (modality gap) :** les similarités texte→texte (~0,65-0,85) sont bien plus élevées que texte→image (~0,25-0,35). Une recherche unique « all » ne renvoie donc jamais d'image : le `retriever_tool` interroge chaque modalité séparément puis fusionne.
- **Agent ReAct :** l'agent appelle l'outil adapté (texte pour la définition, `content_type='image'` pour la figure, `image_retriever_tool` quand un chemin d'image est fourni) et répond sans outil à une salutation. Au test 2, il invente d'abord un chemin d'image ; l'outil renvoie une erreur explicite et l'agent **se corrige** en appelant `retriever_tool(content_type='image')` : c'est la boucle Observation → Reasoning du pattern ReAct.
- **Chaînage image → légende → texte :** pour une image requête, l'outil retrouve la figure la plus proche, puis utilise sa légende pour récupérer les passages explicatifs ; sans cela (première version), le LLM hallucinait le contenu de la figure.
- **Limites :** l'encodeur texte CLIP (77 tokens) reste moins précis qu'un embedding textuel dédié ; le LLM 7B ne décompose pas toujours une question multi-parties (test 3). Pistes : recherche hybride (CLIP + bge/e5 ou BM25), re-ranking, LLM plus grand ou modèle vision (LLaVA, Qwen-VL) pour lire directement les figures.